In [2]:
pip install geopandas libpysal spreg

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 11.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   ---- ----------------------------------- 2.6/22.9 MB 13.7 MB/s eta 0:00:02
   ---------- ----------------------------- 6.0/22.9 MB 14.7 MB/s eta 0:00:02
   ------------------ --------------------- 10.7/22.9 MB 17.2 MB/s eta 0:00:01
   -------------------------- ------------- 14.9/22.9 MB 18.1 MB/s eta 0:00:01
   -------------------------------- ------- 18.6/22.9 MB 17.5 MB/s eta 0:00:01
   ---------------------------------------  22.8/22.9 MB 18.5 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 17.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ----------------------------- ---------- 4.7/6.3 MB 23.7 MB/s eta 0:00:01
   -------

In [2]:
# pip install geopandas libpysal spreg

import pandas as pd
import numpy as np
import geopandas as gpd
import libpysal
from libpysal.weights import Queen
import spreg
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE    = 'analysis_dataset.csv'
SHAPEFILE     = 'tl_2021_us_county/tl_2021_us_county.shp'
OUTPUT_FILE   = 'spatial_regression_results.csv'

# ============================================================
# Step 1: Load analysis dataset
# ============================================================
print("Loading analysis dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})

df_main = df[
    df['mortality_rate'].notna() &
    df['rucc_2023'].notna() &
    df['median_household_income'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

df_main['log_density_rad'] = np.log(df_main['density_radiologist'] + 1)
df_main['income_10k'] = df_main['median_household_income'] / 10000
df_main['rucc_2023'] = pd.to_numeric(df_main['rucc_2023'], errors='coerce')

print(f"Analysis sample: {len(df_main):,} counties")

# ============================================================
# Step 2: Load shapefile and merge
# ============================================================
print("\nLoading shapefile...")
gdf = gpd.read_file(SHAPEFILE)
gdf = gdf.rename(columns={'GEOID': 'FIPS'})

gdf_merged = gdf.merge(df_main, on='FIPS', how='inner')
print(f"Counties after merge with shapefile: {len(gdf_merged):,}")

ct_count = (gdf_merged['state'] == 'CT').sum()
print(f"CT counties in merged data: {ct_count} (should be 0)")

gdf_merged = gdf_merged.sort_values('FIPS').reset_index(drop=True)

# ============================================================
# Step 3: Build Queen contiguity spatial weights matrix
# ============================================================
print("\nBuilding Queen contiguity spatial weights matrix...")
w = Queen.from_dataframe(gdf_merged, silence_warnings=True)
w.transform = 'r'
print(f"Spatial weights: {w.n} observations")
print(f"Average neighbors: {w.mean_neighbors:.2f}")
print(f"Islands (no neighbors): {len(w.islands)}")

# ============================================================
# Step 4: OLS residuals — Moran's I test
# ============================================================
print("\n=== Moran's I Test on OLS Residuals ===")

y = gdf_merged['mortality_rate'].values
X_vars = ['log_density_rad', 'rucc_2023', 'income_10k', 'pct_uninsured_adults']
X = np.column_stack([np.ones(len(gdf_merged))] +
                    [gdf_merged[v].values for v in X_vars])

beta_ols = np.linalg.lstsq(X, y, rcond=None)[0]
resid_ols = y - X @ beta_ols

mi = libpysal.weights.lag_spatial(w, resid_ols)
moran_i = np.corrcoef(resid_ols, mi)[0, 1]

np.random.seed(42)
n_perm = 999
moran_perm = []
for _ in range(n_perm):
    perm = np.random.permutation(resid_ols)
    mi_perm = libpysal.weights.lag_spatial(w, perm)
    moran_perm.append(np.corrcoef(perm, mi_perm)[0, 1])

moran_perm = np.array(moran_perm)
p_value = (np.sum(moran_perm >= moran_i) + 1) / (n_perm + 1)

print(f"Moran's I = {moran_i:.4f}")
print(f"p-value (permutation, 999 runs) = {p_value:.4f}")

# ============================================================
# Step 4b: Lagrange Multiplier Diagnostics (LM test)
# ============================================================
print("\n=== Lagrange Multiplier Diagnostics ===")

y_arr = gdf_merged['mortality_rate'].values.reshape(-1, 1)
X_arr = np.column_stack([gdf_merged[v].values for v in X_vars])

ols_spat = spreg.OLS(
    y_arr,
    X_arr,
    w=w,
    spat_diag=True,
    name_y='mortality_rate',
    name_x=X_vars,
    name_ds='county_analysis'
)

lm_error_stat, lm_error_p = ols_spat.lm_error
lm_lag_stat,   lm_lag_p   = ols_spat.lm_lag

print(f"LM-error = {lm_error_stat:.4f}, p = {lm_error_p:.4f}")
print(f"LM-lag   = {lm_lag_stat:.4f},   p = {lm_lag_p:.4f}")

if lm_error_stat > lm_lag_stat:
    print("→ LM-error > LM-lag: SEM preferred ✓")
else:
    print("→ LM-lag > LM-error: spatial lag preferred")

# ============================================================
# Step 5: Spatial Error Model (SEM)
# ============================================================
print("\n=== Spatial Error Model (SEM) ===")

sem = spreg.ML_Error(
    y_arr,
    X_arr,
    w=w,
    name_y='mortality_rate',
    name_x=X_vars,
    name_ds='county_analysis'
)

print(sem.summary)

# ============================================================
# Step 6: Compare OLS M2 vs SEM
# ============================================================
print("\n=== Comparison: OLS M2 vs Spatial Error Model ===")

ols_beta = -2.2383
sem_beta = sem.betas[1][0]
direction = "✓ CONSISTENT" if (ols_beta < 0) == (sem_beta < 0) else "✗ INCONSISTENT"
magnitude = abs((sem_beta - ols_beta) / ols_beta) * 100

print(f"Direction: {direction}")
print(f"Magnitude change: {magnitude:.1f}%")

# ============================================================
# Step 7: Save results (including LM test)
# ============================================================
results = {
    'Statistic': [
        'OLS M2: β (log_density_rad)',
        'OLS M2: SE',
        'OLS M2: p-value',
        'OLS M2: R²',
        'OLS M2: N',
        'Moran\'s I',
        'Moran\'s I p-value (permutation)',
        'LM-error statistic',
        'LM-error p-value',
        'LM-lag statistic',
        'LM-lag p-value',
        'SEM: β (log_density_rad)',
        'SEM: SE',
        'SEM: p-value',
        'SEM: Pseudo-R²',
        'SEM: N',
        'SEM: Lambda',
        'SEM: Lambda p-value',
    ],
    'Value': [
        -2.2383,
        0.3927,
        '<0.001',
        0.2872,
        3073,
        round(moran_i, 4),
        round(p_value, 4),
        round(lm_error_stat, 4),
        round(lm_error_p, 4),
        round(lm_lag_stat, 4),
        round(lm_lag_p, 4),
        round(sem_beta, 4),
        round(sem.std_err[1], 4),
        '<0.001',
        round(sem.pr2, 4),
        sem.n,
        round(sem.lam, 4),
        round(sem.z_stat[-1][1], 4),
    ]
}

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nResults saved to: {OUTPUT_FILE}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Moran's I       = {moran_i:.4f}, p = {p_value:.4f}")
print(f"LM-error        = {lm_error_stat:.4f}, p = {lm_error_p:.4f}")
print(f"LM-lag          = {lm_lag_stat:.4f},   p = {lm_lag_p:.4f}")
print(f"OLS M2 β        = {ols_beta:.4f}")
print(f"SEM β           = {sem_beta:.4f}")
print(f"Direction: {direction}, magnitude change: {magnitude:.1f}%")
print()
print("Methods sentence:")
print(f"SEM was selected over a spatial lag model based on Lagrange multiplier")
print(f"diagnostics (LM-error = {lm_error_stat:.2f}, p < 0.001; LM-lag = {lm_lag_stat:.2f}, p = {lm_lag_p:.3f}).")
print("\nDONE.")

Loading analysis dataset...
Analysis sample: 3,073 counties

Loading shapefile...
Counties after merge with shapefile: 3,073
CT counties in merged data: 0 (should be 0)

Building Queen contiguity spatial weights matrix...
Spatial weights: 3073 observations
Average neighbors: 5.83
Islands (no neighbors): 4

=== Moran's I Test on OLS Residuals ===
Moran's I = 0.4281
p-value (permutation, 999 runs) = 0.0010

=== Lagrange Multiplier Diagnostics ===
LM-error = 598.7218, p = 0.0000
LM-lag   = 553.8798,   p = 0.0000
→ LM-error > LM-lag: SEM preferred ✓

=== Spatial Error Model (SEM) ===
ML_Error
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
------------------------------------------------------------------------------------
Data set            :county_analysis
Weights matrix      :     unknown
Dependent Variable  :mortality_rate                Number of Observations:        3073
Mean dependent var  :    163.6512                Number of Variables  